# ส่วนที่ 5 · Fine-tune mBART-50 สรุปข่าวภาษาไทย (Abstractive)

**วิธีใช้ (Colab):** Runtime → Change runtime type → **T4 GPU** แล้ว **Run all** · ใช้เวลาประมาณ 1–1.5 ชั่วโมง

ต้องมี 3 ไฟล์ใน Google Drive โฟลเดอร์ `MyDrive/` (มาจาก `data/processed/` ของคนที่ 1)
1. `train_5000.jsonl` – สุ่ม 5,000 ข่าวจาก train กลาง (seed 42) จำนวนเท่ากับที่ mT5 ใช้
2. `val_300.jsonl` – 300 ข่าวแรกของ val กลาง
3. `test.jsonl` – test กลาง (ใช้ 500 ข่าวแรกเหมือนทุกโมเดล)

ผลลัพธ์: `MyDrive/predictions_mbart.csv` → ใส่ในโฟลเดอร์ `predictions/` ของ repo แล้วรัน `07-eval.ipynb` ใหม่

ตั้งค่าให้เทียบกับ mT5 ได้ตรงๆ: จำนวนข่าวเทรน, epoch, ความยาว input/output, beam และฟังก์ชัน ROUGE เหมือนกันทั้งหมด ต่างกันแค่โมเดลและ learning rate

In [ ]:
!pip -q install transformers sentencepiece accelerate rouge_score pythainlp

## 1. ตั้งค่า

In [ ]:
DRIVE = "/content/drive/MyDrive"
TRAIN_JSONL, VAL_JSONL, TEST_JSONL = f"{DRIVE}/train_5000.jsonl", f"{DRIVE}/val_300.jsonl", f"{DRIVE}/test.jsonl"
N_TEST = 500

MODEL_NAME = "facebook/mbart-large-50"
LANG       = "th_TH"
MAX_INPUT, MAX_TARGET = 512, 128      # เท่ากับ mT5
EPOCHS     = 2                        # เท่ากับ mT5
LR         = 3e-5                     # mBART ใช้ lr ต่ำกว่า mT5 (1e-4)
BATCH_SIZE, GRAD_ACC = 4, 4           # batch จริง 16 เท่ากับ mT5
NUM_BEAMS  = 4
SEED       = 42
OUTPUT_DIR = "mbart-thaisum"

## 2. โหลดข้อมูลจาก Drive

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount("/content/drive")
train_df = pd.read_json(TRAIN_JSONL, lines=True)
val_df   = pd.read_json(VAL_JSONL, lines=True)
test_df  = pd.read_json(TEST_JSONL, lines=True).head(N_TEST)
print("train", len(train_df), "| val", len(val_df), "| test", len(test_df))

## 3. Tokenize
mBART-50 ต้องบอกภาษาไทย (`th_TH`) ทั้งฝั่งข่าวและฝั่งบทสรุป

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang=LANG, tgt_lang=LANG)
TH_ID = tokenizer.convert_tokens_to_ids(LANG)

def prep(batch):
    x = tokenizer(batch["body"], max_length=MAX_INPUT, truncation=True)
    x["labels"] = tokenizer(text_target=batch["summary"], max_length=MAX_TARGET, truncation=True)["input_ids"]
    return x

cols = ["body", "summary"]
train_ds = Dataset.from_pandas(train_df[cols]).map(prep, batched=True, remove_columns=cols)
val_ds   = Dataset.from_pandas(val_df[cols]).map(prep, batched=True, remove_columns=cols)
print(train_ds)

## 4. ROUGE ภาษาไทย (ฟังก์ชันเดียวกับทุกคน)

In [ ]:
from rouge_score import rouge_scorer
from pythainlp.tokenize import word_tokenize

class ThaiTokenizer:
    def tokenize(self, text):
        return [w for w in word_tokenize(text, engine="newmm") if w.strip()]

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], tokenizer=ThaiTokenizer())

def thai_rouge(preds, refs):
    s = [scorer.score(r, p) for p, r in zip(preds, refs)]
    return {k: round(100 * sum(x[k].fmeasure for x in s) / len(s), 2) for k in ["rouge1", "rouge2", "rougeL"]}

## 5. Fine-tune
> mBART เทรนแบบ fp16 ได้ (ต่างจาก mT5) จึงเร็วกว่า · ถ้า `CUDA out of memory` ให้ลด `BATCH_SIZE` เป็น 2 และเพิ่ม `GRAD_ACC` เป็น 8

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq, set_seed
set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if device == "cuda" else "ไม่มี (ช้ามาก ให้เปิด T4 GPU)")

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.generation_config.forced_bos_token_id = TH_ID

args = Seq2SeqTrainingArguments(
    output_dir="checkpoints", num_train_epochs=EPOCHS, learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC, weight_decay=0.01, warmup_steps=30,
    optim="adafactor", gradient_checkpointing=True, fp16=(device == "cuda"),
    eval_strategy="epoch", save_strategy="no", logging_steps=50,
    report_to="none", seed=SEED)

trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                         data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
                         processing_class=tokenizer)
train_result = trainer.train()
train_result

## 6. วัดผลบน test กลาง 500 ข่าว

In [ ]:
def summarize(texts, bs=8):
    out = []
    m = trainer.model.eval()
    for i in range(0, len(texts), bs):
        enc = tokenizer(texts[i:i+bs], max_length=MAX_INPUT, truncation=True, padding=True, return_tensors="pt").to(m.device)
        with torch.no_grad():
            ids = m.generate(**enc, max_new_tokens=MAX_TARGET, num_beams=NUM_BEAMS, no_repeat_ngram_size=3,
                             length_penalty=1.0, forced_bos_token_id=TH_ID)
        out += tokenizer.batch_decode(ids, skip_special_tokens=True)
        print(len(out), "/", len(texts))
    return out

preds = summarize(test_df.body.tolist())
scores = thai_rouge(preds, test_df.summary.tolist())
result = {"model": MODEL_NAME, **scores, "train_size": len(train_df), "test_size": len(test_df),
          "epochs": EPOCHS, "lr": LR, "max_input": MAX_INPUT, "num_beams": NUM_BEAMS,
          "training_loss": round(train_result.training_loss, 4)}
result

## 7. บันทึกผล (ส่งให้คนที่ 6 และ 7)

In [ ]:
import json, shutil
out = test_df[["id", "type", "body", "summary"]].assign(pred=preds)
out.to_csv("predictions_mbart.csv", index=False)
json.dump(result, open("result_mbart.json", "w"), ensure_ascii=False, indent=2)
shutil.copy("predictions_mbart.csv", f"{DRIVE}/predictions_mbart.csv")
shutil.copy("result_mbart.json", f"{DRIVE}/result_mbart.json")
print("บันทึกแล้ว: MyDrive/predictions_mbart.csv และ result_mbart.json")
pd.set_option("display.max_colwidth", 150)
out.head(3)[["summary", "pred"]]

## 8. (ไม่บังคับ) เก็บโมเดลไว้ใน Drive
โมเดลใหญ่ประมาณ 2.4 GB · เอาคอมเมนต์ออกถ้าต้องการใช้ต่อ

In [ ]:
# trainer.save_model(OUTPUT_DIR); tokenizer.save_pretrained(OUTPUT_DIR)
# shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR); shutil.copy(OUTPUT_DIR + ".zip", DRIVE)